# 02. Tiền xử lý & Thiết kế Data Warehouse


- Gộp các bảng dữ liệu gốc của Olist, lọc đơn `delivered` và làm sạch.
- Tách thành các bảng Dim/Fact chuẩn hóa để phục vụ làm Cube và gom cụm.

In [1]:
import os
import pandas as pd

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
print("Đã đọc xong data raw")

Đã đọc xong data raw


## Groupby Payment và Review để tránh trùng dòng khi merge
- Payments: Tính tổng tiền và lấy phương thức thanh toán xuất hiện nhiều nhất.
- Reviews: Tính điểm đánh giá trung bình theo từng đơn.

In [2]:
payment_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode().iloc[0] if not x.mode().empty else 'unknown')
).reset_index()

review_agg = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean')
).reset_index()

print("Đã aggregate xong payment và review.")

Đã aggregate xong payment và review.


## Merge và làm sạch data (Tạo final_sales)
- Chỉ lọc các đơn có trạng thái `delivered`.
- Tính `delivery_days` (Ngày nhận - Ngày đặt), chuyển các giá trị âm về Null rồi fill bằng median.
- Tính `total_amount` = price + freight_value.
- Fill các cột bị missing bằng 'unknown' hoặc median.

In [3]:
df = pd.merge(orders, customers, on='customer_id', how='inner')

# Chỉ lấy đơn đã giao thành công
df = df[df['order_status'] == 'delivered'].copy()

df = pd.merge(df, order_items, on='order_id', how='inner')
df = pd.merge(df, products, on='product_id', how='left')
df = pd.merge(df, sellers, on='seller_id', how='left')
df = pd.merge(df, payment_agg, on='order_id', how='left')
df = pd.merge(df, review_agg, on='order_id', how='left')

# Ép kiểu datetime kèm errors='coerce' để bẫy lỗi
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

df = df.drop_duplicates()

df['total_amount'] = df['price'] + df['freight_value']
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Tính delivery_days, xử lý ngày âm và fill khuyết bằng median
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df.loc[df['delivery_days'] < 0, 'delivery_days'] = pd.NA
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())

# Giữ đúng 15 cột yêu cầu cho final_sales
final_cols = [
    'order_id', 'customer_id', 'customer_unique_id', 'product_id', 'seller_id',
    'order_purchase_timestamp', 'order_month', 'customer_state', 'seller_state',
    'product_category_name', 'payment_type', 'price', 'freight_value',
    'total_amount', 'review_score', 'delivery_days'
]
final_sales = df[final_cols].copy()

# Fill missing values
final_sales['product_category_name'] = final_sales['product_category_name'].fillna('unknown')
final_sales['payment_type'] = final_sales['payment_type'].fillna('unknown')
final_sales['review_score'] = final_sales['review_score'].fillna(final_sales['review_score'].median())

os.makedirs('../data/processed', exist_ok=True)
final_sales.to_csv('../data/processed/final_sales.csv', index=False)
print("Đã xuất file final_sales.csv")

Đã xuất file final_sales.csv


## Tách bảng theo mô hình Star Schema
- Dimension dùng đúng khóa chính của đối tượng (customer_id, product_id, seller_id).
- `dim_date` chỉ lưu danh sách các tháng (`order_month`) duy nhất.
- `fact_sales` giữ lại `payment_type` và `order_month` để link với các bảng chiều.

In [4]:
os.makedirs('../data/warehouse', exist_ok=True)

dim_customer = final_sales[['customer_id', 'customer_unique_id', 'customer_state']].drop_duplicates()
dim_customer.to_csv('../data/warehouse/dim_customer.csv', index=False)

dim_product = final_sales[['product_id', 'product_category_name']].drop_duplicates()
dim_product.to_csv('../data/warehouse/dim_product.csv', index=False)

dim_seller = final_sales[['seller_id', 'seller_state']].drop_duplicates()
dim_seller.to_csv('../data/warehouse/dim_seller.csv', index=False)

dim_payment = final_sales[['payment_type']].drop_duplicates()
dim_payment.to_csv('../data/warehouse/dim_payment.csv', index=False)

# dim_date không dùng order_id nữa
dim_date = final_sales[['order_month']].drop_duplicates().sort_values('order_month')
dim_date.to_csv('../data/warehouse/dim_date.csv', index=False)

# Bảng Fact chứa các cột số liệu và khóa ngoại
fact_sales = final_sales[[
    'order_id', 'customer_id', 'product_id', 'seller_id', 'payment_type', 'order_month',
    'order_purchase_timestamp', 'price', 'freight_value', 'total_amount', 'review_score', 'delivery_days'
]].drop_duplicates()
fact_sales.to_csv('../data/warehouse/fact_sales.csv', index=False)

print("Đã tạo xong các bảng Dim và Fact.")

Đã tạo xong các bảng Dim và Fact.


## Kiểm tra chất lượng dữ liệu đầu ra (Validation)
Check nhanh kích thước, missing value, trùng lặp và lỗi ngày âm.

In [5]:
print("Shape:", final_sales.shape)

print("\nMissing values từng cột:")
print(final_sales.isnull().sum())

print("\nSố dòng trùng lặp hoàn toàn:", final_sales.duplicated().sum())

print("Số dòng có delivery_days bị âm:", (final_sales['delivery_days'] < 0).sum())

Shape: (110197, 16)

Missing values từng cột:
order_id                    0
customer_id                 0
customer_unique_id          0
product_id                  0
seller_id                   0
order_purchase_timestamp    0
order_month                 0
customer_state              0
seller_state                0
product_category_name       0
payment_type                0
price                       0
freight_value               0
total_amount                0
review_score                0
delivery_days               0
dtype: int64

Số dòng trùng lặp hoàn toàn: 10001
Số dòng có delivery_days bị âm: 0
